In [ ]:
# Telegram to Google Drive transfer (single Colab cell)
# Input formats supported for the Telegram message:
#   1) Public message link:  https://t.me/username/123
#   2) Private/supergroup link: https://t.me/c/123456789/123
#   3) Bare message ID:       123  (you will also be prompted for the chat username/link/ID)
# The script downloads Telegram attachments to a temporary folder, then copies them to Google Drive.
# It never deletes Telegram files and asks before overwriting existing Drive files.

import importlib.util
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

if importlib.util.find_spec('telethon') is None:
    print('Installing Telethon...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'telethon'])

from google.colab import drive
from telethon import TelegramClient
from telethon.errors import SessionPasswordNeededError
from telethon.tl.types import Message

def ask_required(prompt):
    value = input(prompt).strip()
    if not value:
        raise ValueError(f'Missing required value: {prompt}')
    return value

def parse_message_input(raw):
    raw = raw.strip()
    public_match = re.fullmatch(r'https?://t\.me/([A-Za-z0-9_]{5,})/(\d+)', raw)
    private_match = re.fullmatch(r'https?://t\.me/c/(\d+)/(\d+)', raw)
    id_match = re.fullmatch(r'\d+', raw)
    if public_match:
        return public_match.group(1), int(public_match.group(2))
    if private_match:
        # Telethon represents t.me/c/<internal_id>/... chats as -100<internal_id>.
        return int('-100' + private_match.group(1)), int(private_match.group(2))
    if id_match:
        chat = ask_required('Enter Telegram chat username, invite/link, numeric chat ID, or phone/contact for this message ID: ')
        return chat, int(raw)
    raise ValueError(
        "Invalid Telegram message input. Use 'https://t.me/username/123', "
        "'https://t.me/c/123456789/123', or a numeric message ID like '123'."
    )

def safe_destination(dest_dir, filename):
    dest = dest_dir / filename
    if dest.exists():
        answer = input(f'File exists in Drive: {dest}. Overwrite? Type YES to overwrite: ').strip()
        if answer != 'YES':
            stem, suffix = dest.stem, dest.suffix
            counter = 1
            while dest.exists():
                dest = dest_dir / f'{stem} ({counter}){suffix}'
                counter += 1
            print(f'Will avoid overwrite and copy as: {dest.name}')
    return dest

print('Mounting Google Drive...')
drive.mount('/content/drive')

api_id_text = ask_required('Enter Telegram API ID: ')
if not api_id_text.isdigit():
    raise ValueError('Telegram API ID must be numeric.')
api_id = int(api_id_text)
api_hash = ask_required('Enter Telegram API hash: ')
phone = ask_required('Enter Telegram phone number, including country code (for example +15551234567): ')
message_input = ask_required(
    "Enter Telegram message link or ID (formats: 'https://t.me/username/123', "
    "'https://t.me/c/123456789/123', or '123'): "
)
drive_folder_text = ask_required('Enter target Google Drive folder path (for example /content/drive/MyDrive/TelegramDownloads): ')

chat_ref, message_id = parse_message_input(message_input)
drive_folder = Path(drive_folder_text).expanduser()
if not str(drive_folder.resolve()).startswith('/content/drive/'):
    raise ValueError('Target folder must be inside the mounted Google Drive path: /content/drive/...')
drive_folder.mkdir(parents=True, exist_ok=True)

async def main():
    with tempfile.TemporaryDirectory() as tmp:
        tmp_dir = Path(tmp)
        session_path = str(tmp_dir / 'telegram_colab_session')
        client = TelegramClient(session_path, api_id, api_hash)
        await client.connect()
        if not await client.is_user_authorized():
            print('Authorizing Telegram...')
            await client.send_code_request(phone)
            code = ask_required('Enter the Telegram login code you received: ')
            try:
                await client.sign_in(phone=phone, code=code)
            except SessionPasswordNeededError:
                password = ask_required('Two-step verification enabled. Enter Telegram password: ')
                await client.sign_in(password=password)

        print('Fetching Telegram message...')
        message = await client.get_messages(chat_ref, ids=message_id)
        if not isinstance(message, Message):
            raise ValueError('Could not find that Telegram message. Check the link/ID and account access.')
        if not message.media:
            raise ValueError('The selected Telegram message has no downloadable attachment.')

        print('Downloading from Telegram...')
        downloaded = await client.download_media(message, file=str(tmp_dir) + os.sep)
        if not downloaded:
            raise RuntimeError('Download did not return a file path.')
        downloaded_path = Path(downloaded)
        if not downloaded_path.exists() or downloaded_path.stat().st_size == 0:
            raise RuntimeError('Downloaded file is missing or empty.')

        print('Copying to Google Drive...')
        destination = safe_destination(drive_folder, downloaded_path.name)
        shutil.copy2(downloaded_path, destination)
        if not destination.exists() or destination.stat().st_size != downloaded_path.stat().st_size:
            raise RuntimeError('Copy verification failed: destination file is missing or size differs.')

        await client.disconnect()
        print(f'Success! Transferred {downloaded_path.name} to {destination}')

await main()
